# Test Activity Tracking API

This notebook tests the activity tracking feature for monitoring user activities detected by AI cameras.

**Features tested:**
- Recording user activities (phone usage, sleeping, not focusing, talking)
- Uploading proof images with activity records
- Querying current activities with filters
- Getting activity statistics
- Getting activity history

**Before running:**
1. Make sure backend is running
2. Have test users and cameras configured
3. Login as ORG ADMIN

## Setup

In [ ]:
import requests
import os
import io
import cv2
import numpy as np
from datetime import datetime, timezone
from dotenv import load_dotenv
import time

load_dotenv()

BACKEND_URL = os.getenv('SO_BACKEND_API_URL', 'http://localhost:7091')
session = requests.Session()
session.headers.update({"accept": "application/json"})

print('Setup complete!')
print(f'Backend API: {BACKEND_URL}')

## Login

In [ ]:
def login_to_backend(email=None, password=None, client_slug='humblebee'):
    if email is None:
        email = os.getenv('SO_ADMIN_EMAIL', 'admin@humblebee.ai')
    if password is None:
        password = os.getenv('SO_ADMIN_PASSWORD', 'admin123')

    print(f'Logging in as {email} to org "{client_slug}"...')

    response = session.post(
        f'{BACKEND_URL}/api/auth/login',
        json={'email': email, 'password': password, 'client_slug': client_slug}
    )

    if response.status_code == 200:
        data = response.json()
        token = data.get('token') or data.get('accessToken')
        session.headers.update({'Authorization': f'Bearer {token}'})
        print('Login successful')
        print(f"  User: {data.get('user', {}).get('full_name')}")
        return token, client_slug
    else:
        print(f'Login failed: {response.status_code}')
        print(response.text[:200])
        return None, None

# Update with your credentials
auth_token, slug = login_to_backend(
    email='humblebee@gmail.com',
    password='Humblebee2025@',
    client_slug='humblebee'
)

## Get Users and Cameras

In [ ]:
def list_org_users(slug, page=1, limit=50):
    r = session.get(f"{BACKEND_URL}/api/org/{slug}/users", params={"page": page, "limit": limit})
    r.raise_for_status()
    return r.json()

def list_org_cameras(slug):
    r = session.get(f"{BACKEND_URL}/api/org/{slug}/cameras")
    r.raise_for_status()
    return r.json()

users = list_org_users(slug)
cameras = list_org_cameras(slug)

print(f'Found {len(users)} users and {len(cameras)} cameras')
print(f'\nSample users:')
for u in users[:5]:
    print(f"  - {u.get('full_name')} (ID: {u.get('id')})")
print(f'\nSample cameras:')
for c in cameras[:3]:
    print(f"  - {c.get('name')} (ID: {c.get('id')}) - {c.get('location', 'No location')}")

## Helper: Create Test Proof Image

In [ ]:
def create_test_image(text="ACTIVITY DETECTED", activity_icon="📱", size=(640, 480)):
    """Create a test proof image with text overlay"""
    # Create blue-ish background (camera-like)
    img = np.zeros((size[1], size[0], 3), dtype=np.uint8)
    img[:, :] = [40, 60, 100]  # Dark blue

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Add text overlays
    cv2.putText(img, activity_icon, (50, 80), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 3)
    cv2.putText(img, text, (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
    cv2.putText(img, f"Timestamp: {timestamp}", (50, 250), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 1)
    cv2.putText(img, "PROOF IMAGE", (50, 320), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (100, 200, 100), 2)

    # Encode to JPEG
    _, encoded = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return encoded.tobytes()

# Test image creation
test_img = create_test_image()
print(f'Test image created: {len(test_img)} bytes')

## Helper: ISO Timestamp

In [ ]:
def iso_now():
    """Return current timestamp in ISO format with Z suffix"""
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

print(f'Current timestamp: {iso_now()}')

## Test 1: Record Activity (JSON Only)

Simple activity recording without proof image

In [ ]:
def record_activity_json(slug, user_id, camera_id, activity_type, confidence_score=0.95):
    """
    Record a user activity detection (JSON only, no proof image)

    Args:
        slug: Organization slug
        user_id: User ID
        camera_id: Camera ID where activity was detected
        activity_type: One of: phone_usage, sleeping, not_focusing, talking, unknown
        confidence_score: AI confidence (0.0 to 1.0)
    """
    url = f"{BACKEND_URL}/api/org/{slug}/activities"
    timestamp = iso_now()

    payload = {
        'user_id': int(user_id),
        'camera_id': int(camera_id),
        'activity_type': activity_type,
        'timestamp': timestamp,
        'confidence_score': confidence_score
    }

    session.headers.update({'Content-Type': 'application/json'})
    r = session.post(url, json=payload)

    if r.status_code == 201:
        result = r.json()
        print(f'✅ Activity recorded: {activity_type} for user {user_id}')
        print(f'   ID: {result.get("id")}')
        print(f'   Timestamp: {result.get("timestamp")}')
        return result
    else:
        print(f'❌ Failed: {r.status_code}')
        print(f'   Error: {r.text[:300]}')
        return None

# ====== CONFIGURE TEST USER HERE ======
# Change this to test with a specific user ID, or leave as None to use first user from list
test_user_id = 68  # Example: test_user_id = 81
# =====================================

if users and cameras:
    user_id = test_user_id if test_user_id is not None else users[0]['id']
    camera = cameras[0]  # Always use first camera from list

    user_name = f"User ID {user_id}" if test_user_id is not None else users[0]['full_name']

    print(f'Testing with: {user_name} @ {camera["name"]}')
    result = record_activity_json(slug, user_id, camera['id'], 'phone_usage', 0.92)

## Test 2: Record Activity with Proof Image

This is how the AI system should send activity data with proof images (camera frames)

In [ ]:
def record_activity_with_proof(slug, user_id, camera_id, activity_type, proof_image_data, confidence_score=0.95, metadata=None):
    """
    Record activity with proof image (multipart/form-data)

    Args:
        slug: Organization slug
        user_id: User ID
        camera_id: Camera ID
        activity_type: phone_usage, sleeping, not_focusing, talking, unknown
        proof_image_data: Image bytes (JPEG/PNG)
        confidence_score: AI confidence (0.0 to 1.0)
        metadata: Optional dict with extra data
    """
    url = f"{BACKEND_URL}/api/org/{slug}/activities"
    timestamp = iso_now()

    # Prepare multipart form data
    files = {'proof_image': ('proof.jpg', io.BytesIO(proof_image_data), 'image/jpeg')}
    data = {
        'user_id': str(user_id),
        'camera_id': str(camera_id),
        'activity_type': activity_type,
        'timestamp': timestamp,
        'confidence_score': str(confidence_score)
    }

    if metadata:
        import json
        data['metadata'] = json.dumps(metadata)

    # Remove Content-Type header for multipart
    headers = {k: v for k, v in session.headers.items() if k.lower() != 'content-type'}
    r = requests.post(url, headers=headers, files=files, data=data)

    if r.status_code == 201:
        result = r.json()
        print(f'✅ Activity with proof recorded: {activity_type}')
        print(f'   ID: {result.get("id")}')
        print(f'   User: {result.get("user_id")}')
        proof_url = result.get("proof_image_url") or "N/A"
        print(f'   Proof URL: {proof_url if len(proof_url) <= 80 else proof_url[:80] + "..."}')
        return result
    else:
        print(f'❌ Failed: {r.status_code}')
        print(f'   Error: {r.text[:300]}')
        return None

# ====== CONFIGURE TEST USER HERE ======
# Change this to test with a specific user ID, or leave as None to use first user from list
test_user_id = 54  # Example: test_user_id = 82
# =====================================

if users and cameras:
    user_id = test_user_id if test_user_id is not None else users[0]['id']
    camera = cameras[0]  # Always use first camera from list

    user_name = f"User ID {user_id}" if test_user_id is not None else users[0]['full_name']

    print(f'Testing with: {user_name} @ {camera["name"]}')

    proof_image = create_test_image(f"PHONE USAGE - {user_name}", "📱")
    metadata = {
        'detection_method': 'yolo_v8',
        'model_version': '1.0.3',
        'hand_position': 'right'
    }

    result = record_activity_with_proof(
        slug,
        user_id,
        camera['id'],
        'phone_usage',
        proof_image,
        confidence_score=0.94,
        metadata=metadata
    )

## Test 3: Record Multiple Activities

Simulate multiple users performing different activities

In [ ]:
# ====== CONFIGURE TEST USERS HERE ======
# You can manually set user IDs here for testing
# Leave as None to use users from the list automatically
manual_user_ids = [49, 44, 41, 64, 57]  # Change to [81, 82, 83, etc.] to test specific users
# ========================================

activity_scenarios = [
    ('phone_usage', '📱', 0.92, True),
    ('sleeping', '😴', 0.88, True),
    ('not_focusing', '⚠️', 0.85, False),
    ('talking', '💬', 0.90, True),
    ('phone_usage', '📱', 0.95, True),
]

print('Recording various activities...')
print('=' * 70)

created_ids = []
camera = cameras[0]  # Always use first camera from list

for i, (activity_type, icon, confidence, with_image) in enumerate(activity_scenarios):
    if i >= len(users):
        print(f'\n⚠️  Skipping scenario {i+1} - not enough users in list')
        break

    # Use manual user ID if provided, otherwise use from list
    if manual_user_ids[i] is not None:
        user_id = manual_user_ids[i]
        user_name = f"User ID {user_id}"
        print(f'\n{i+1}. {icon} {activity_type} - {user_name} @ {camera["name"]} (manual ID)')
    else:
        user = users[i % len(users)]
        user_id = user['id']
        user_name = user['full_name']
        print(f'\n{i+1}. {icon} {activity_type} - {user_name} @ {camera["name"]}')

    if with_image:
        proof_image = create_test_image(f"{activity_type.upper()} - {user_name}", icon)
        result = record_activity_with_proof(
            slug, user_id, camera['id'], activity_type, proof_image, confidence
        )
    else:
        result = record_activity_json(
            slug, user_id, camera['id'], activity_type, confidence
        )

    if result:
        created_ids.append(result.get('id'))

    time.sleep(0.5)  # Small delay between requests

print('\n' + '=' * 70)
print(f'✅ Created {len(created_ids)} activity records!')
print('Check the Activities page in the frontend to see them!')
print('\n💡 TIP: To test with specific users, edit manual_user_ids at the top of this cell')
print('   Example: manual_user_ids = [81, 82, 83, 84, 85]')

## Test 4: Get Current Activities

Query activities detected in the last 30 minutes

In [ ]:
def get_current_activities(slug, activity_types=None, stale_minutes=30):
    """
    Get current activities

    Args:
        slug: Organization slug
        activity_types: Comma-separated activity types to filter (optional)
        stale_minutes: Consider activities older than this as stale (default 30)
    """
    params = {'stale_minutes': stale_minutes}
    if activity_types:
        params['activity_types'] = activity_types

    r = session.get(f"{BACKEND_URL}/api/org/{slug}/activities/current", params=params)
    r.raise_for_status()
    return r.json()

# Get all current activities
print('Getting all current activities (last 30 min)...')
activities = get_current_activities(slug)
print(f'\n✅ Found {len(activities)} current activities:\n')

icon_map = {
    'phone_usage': '📱',
    'sleeping': '😴',
    'not_focusing': '⚠️',
    'talking': '💬',
    'unknown': '❓'
}

for act in activities:
    icon = icon_map.get(act['activity_type'], '❓')
    print(f"{icon} {act['full_name']}: {act['activity_type']}")
    print(f"   Camera: {act['camera_name']}")
    print(f"   Detected: {act['detected_at']} ({act.get('minutes_ago', 0):.1f} min ago)")
    if act.get('confidence_score'):
        print(f"   Confidence: {act['confidence_score']*100:.1f}%")
    if act.get('proof_image_url'):
        print(f"   Proof: {act['proof_image_url'][:60]}...")
    print()

# Get only phone usage activities
print('\n' + '=' * 70)
print('Getting ONLY phone usage activities...')
phone_activities = get_current_activities(slug, activity_types='phone_usage')
print(f'\n✅ Found {len(phone_activities)} phone usage activities')
for act in phone_activities:
    print(f"  📱 {act['full_name']} @ {act['camera_name']}")

## Test 5: Get Activity Statistics

In [ ]:
def get_activity_stats(slug, stale_minutes=30):
    """Get aggregated activity statistics"""
    params = {'stale_minutes': stale_minutes}
    r = session.get(f"{BACKEND_URL}/api/org/{slug}/activities/stats", params=params)
    r.raise_for_status()
    return r.json()

stats = get_activity_stats(slug)

print('Activity Statistics (Last 30 minutes):')
print('=' * 70)

for stat in stats.get('stats', []):
    icon = icon_map.get(stat['activity_type'], '❓')
    print(f"\n{icon} {stat['activity_type'].upper().replace('_', ' ')}")
    print(f"   Unique Users: {stat['unique_users']}")
    print(f"   Total Detections: {stat['detection_count']}")

if not stats.get('stats'):
    print('\nNo activity statistics found (no recent activities)')

## Test 6: Get Activity History

Query historical activity records with filters

In [ ]:
def get_activity_history(slug, user_id=None, activity_types=None, page=1, limit=20):
    """
    Get activity history with optional filters

    Args:
        slug: Organization slug
        user_id: Filter by user ID (optional)
        activity_types: Comma-separated activity types to filter (optional)
        page: Page number
        limit: Items per page
    """
    params = {'page': page, 'limit': limit}
    if user_id:
        params['user_id'] = user_id
    if activity_types:
        params['activity_types'] = activity_types

    r = session.get(f"{BACKEND_URL}/api/org/{slug}/activities/history", params=params)
    r.raise_for_status()
    return r.json()

# Get recent activity history
print('Getting activity history (last 10 records)...')
history = get_activity_history(slug, limit=10)

print(f'\n✅ Found {len(history.get("records", []))} records\n')

for i, record in enumerate(history.get('records', []), 1):
    icon = icon_map.get(record['activity_type'], '❓')
    print(f"{i}. {icon} {record['full_name']}: {record['activity_type']}")
    print(f"   Camera: {record['camera_name']}")
    print(f"   Time: {record['timestamp']}")
    if record.get('confidence_score'):
        print(f"   Confidence: {record['confidence_score']*100:.1f}%")
    if record.get('proof_image_url'):
        print(f"   Proof: {record['proof_image_url'][:60]}...")
    print()

# ====== CONFIGURE TEST USER HERE ======
# Change this to filter history for a specific user, or leave as None to use first user
filter_user_id = None  # Example: filter_user_id = 81
# =====================================

# Get history for specific user
if users:
    target_user_id = filter_user_id if filter_user_id is not None else users[0]['id']
    target_user_name = f"User ID {target_user_id}" if filter_user_id is not None else users[0]["full_name"]

    print('\n' + '=' * 70)
    print(f'Getting history for {target_user_name}...')
    user_history = get_activity_history(slug, user_id=target_user_id, limit=5)
    print(f'✅ Found {len(user_history.get("records", []))} records for this user')

    for i, record in enumerate(user_history.get('records', []), 1):
        icon = icon_map.get(record['activity_type'], '❓')
        print(f"  {i}. {icon} {record['activity_type']} at {record['timestamp'][11:19]}")

## Summary

### API Endpoints for AI Team:

**1. Record Activity (POST) - JSON Only**
```http
POST /api/org/{slug}/activities
Content-Type: application/json
Authorization: Bearer {token}

{
  "user_id": 123,
  "camera_id": 5,
  "activity_type": "phone_usage",
  "timestamp": "2025-11-26T10:30:45.123Z",
  "confidence_score": 0.92
}
```

**2. Record Activity with Proof Image (POST) - Multipart**
```http
POST /api/org/{slug}/activities
Content-Type: multipart/form-data
Authorization: Bearer {token}

Form fields:
  - user_id: "123"
  - camera_id: "5"
  - activity_type: "phone_usage"
  - timestamp: "2025-11-26T10:30:45.123Z"
  - confidence_score: "0.92"
  - metadata: '{"key": "value"}' (optional JSON string)
  - proof_image: (binary file data)
```

**Activity Types:**
- `phone_usage` - User using phone 📱
- `sleeping` - User sleeping 😴
- `not_focusing` - User not focused on work ⚠️
- `talking` - User talking to someone 💬
- `unknown` - Unknown activity ❓

**3. Get Current Activities (GET)**
```http
GET /api/org/{slug}/activities/current?activity_types=phone_usage,sleeping&stale_minutes=30
```

**4. Get Statistics (GET)**
```http
GET /api/org/{slug}/activities/stats?stale_minutes=30
```

**5. Get History (GET)**
```http
GET /api/org/{slug}/activities/history?user_id=123&activity_types=phone_usage&page=1&limit=50
```

**Note:** Currently only `phone_usage` is active in the frontend. Other activity types are available in the API but hidden in the UI until fully implemented.